# 09 — Avaliação

**Objetivo:** selecionar o modelo campeão com base em discriminação, separação, ganho de negócio e estabilidade.

**Métricas:**
* **AUC-ROC** e **KS** (discriminação/separação)
* **Brier Score** e curva de calibração (qualidade das probabilidades)
* **Lift por decisão** e ganhos acumulados (valor de negócio)
* **Estabilidade temporal** — AUC por snapshot de validação/teste (out-of-time)
* **Interpretabilidade** — importância de variáveis do campeão

**Output:** `model_evaluation` + definição do modelo campeão.

In [0]:
# ── Configuração central (idem notebook 00) ──
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss, precision_score, recall_score, f1_score

preds = spark.table(f"{CATALOG}.{SCHEMA}.model_predictions").toPandas()
pred_val  = preds[preds["base"] == "validacao"]
pred_test = preds[preds["base"] == "teste"]

print("Validação:", len(pred_val), "| Teste OOT:", len(pred_test))

def ks_statistic(pdf):
    pdf = pdf.sort_values("score", ascending=False).reset_index(drop=True)
    cum_ev  = pdf["target"].cumsum() / pdf["target"].sum()
    cum_nev = (1 - pdf["target"]).cumsum() / (1 - pdf["target"]).sum()
    return float((cum_ev - cum_nev).abs().max())

In [0]:
# 1) Painel comparativo de modelos (validação e teste)
linhas = []
for nome, g_val in pred_val.groupby("modelo"):
    g_test = pred_test[pred_test["modelo"] == nome]
    linhas.append({
        "modelo": nome,
        "auc_validacao": round(roc_auc_score(g_val["target"], g_val["score"]), 4),
        "ks_validacao": round(ks_statistic(g_val), 4),
        "auc_teste": round(roc_auc_score(g_test["target"], g_test["score"]), 4),
        "ks_teste": round(ks_statistic(g_test), 4),
        "brier_teste": round(brier_score_loss(g_test["target"], g_test["score"]), 4),
        "decaimento_auc": round(roc_auc_score(g_val["target"], g_val["score"]) - roc_auc_score(g_test["target"], g_test["score"]), 4),
    })

avaliacao = pd.DataFrame(linhas).sort_values("auc_teste", ascending=False)
display(avaliacao)

# Critério de campeão: maior AUC no TESTE, monitorando decaimento vs. validação
campeao = avaliacao.iloc[0]["modelo"]
print(f"🏆 Modelo campeão (AUC no teste): {campeao}")

In [0]:
# 2) Lift por decisão e ganhos acumulados (modelo campeão, base de teste)
base_camp = pred_test[pred_test["modelo"] == campeao].copy()
base_camp["decis"] = pd.qcut(base_camp["score"].rank(method="first"), 10,
                             labels=[f"D{d}" for d in range(10, 0, -1)])
taxa_global = base_camp["target"].mean()

decis = (base_camp.groupby("decis", observed=True)
         .agg(n=("target", "count"), eventos=("target", "sum"), score_medio=("score", "mean"))
         .reset_index())
decis["taxa_evento"] = (decis["eventos"] / decis["n"]).round(4)
decis["lift"] = (decis["taxa_evento"] / taxa_global).round(2)
decis["eventos_acumulados_%"] = (decis["eventos"].cumsum() / decis["eventos"].sum() * 100).round(1)
display(decis)

print(f"Taxa global de evento: {taxa_global:.2%}")
print(f"Lift no 1º decisão: {decis.iloc[0]['lift']}x")

In [0]:
# 3) Calibração das probabilidades (campeão, teste)
base_camp["bin_score"] = pd.qcut(base_camp["score"].rank(method="first"), 10, labels=False)
calibracao = (base_camp.groupby("bin_score")
              .agg(n=("target", "count"), score_medio=("score", "mean"), taxa_observada=("target", "mean"))
              .reset_index())
display(calibracao.round(4))
print("Brier Score (0 = perfeito):", round(brier_score_loss(base_camp["target"], base_camp["score"]), 4))

# 4) Precision/Recall/F1 no ponto de corte da prevalência (threshold = taxa global)
th = taxa_global
base_camp["classe_pred"] = (base_camp["score"] >= th).astype(int)
print(f"\nPonto de corte = prevalência ({th:.3f}):")
print("Precision:", round(precision_score(base_camp["target"], base_camp["classe_pred"]), 4))
print("Recall:   ", round(recall_score(base_camp["target"], base_camp["classe_pred"]), 4))
print("F1:       ", round(f1_score(base_camp["target"], base_camp["classe_pred"]), 4))

In [0]:
# 5) Estabilidade out-of-time: AUC por snapshot (validação + teste)
oot = []
for (modelo, data), g in preds.groupby(["modelo", "reference_date"]):
    if g["target"].nunique() > 1:
        oot.append({
            "modelo": modelo,
            "reference_date": str(data),
            "auc": round(roc_auc_score(g["target"], g["score"]), 4),
            "n": len(g),
        })
display(pd.DataFrame(oot).sort_values(["modelo", "reference_date"]))

In [0]:
# 6) Interpretabilidade: importância de variáveis do campeão (refit rápido)
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (LogisticRegression, DecisionTreeClassifier,
                                       RandomForestClassifier, GBTClassifier)

train_df = spark.table(f"{CATALOG}.{SCHEMA}.train_base")
feat_cols = [c for c in train_df.columns if c not in ("customerID", "reference_date", "target")]

assembler = VectorAssembler(inputCols=feat_cols, outputCol="raw_features", handleInvalid="keep")
scaler    = StandardScaler(inputCol="raw_features", outputCol="features")
algo = {
    "regressao_logistica": LogisticRegression(featuresCol="features", labelCol="target", maxIter=100),
    "arvore_decisao":      DecisionTreeClassifier(featuresCol="features", labelCol="target", seed=42),
    "random_forest":       RandomForestClassifier(featuresCol="features", labelCol="target", numTrees=200, seed=42),
    "gbt":                 GBTClassifier(featuresCol="features", labelCol="target", maxIter=50, seed=42),
}[campeao]

model_camp = Pipeline(stages=[assembler, scaler, algo]).fit(train_df)
estagio_final = model_camp.stages[-1]

if hasattr(estagio_final, "featureImportances"):
    importancia = pd.DataFrame({
        "feature": feat_cols,
        "importancia": list(estagio_final.featureImportances.toArray()),
    }).sort_values("importancia", ascending=False)
    display(importancia.assign(importancia=lambda d: d["importancia"].round(4)))
else:
    # Regressão Logística: coeficientes padronizados (o sinal indica direção do efeito)
    coeficientes = pd.DataFrame({
        "feature": feat_cols,
        "coeficiente": list(estagio_final.coefficients.toArray()),
    }).sort_values("coeficiente", key=abs, ascending=False)
    display(coeficientes.assign(coeficiente=lambda d: d["coeficiente"].round(4)))

In [0]:
# 7) Gravação do model_evaluation
(spark.createDataFrame(avaliacao.assign(campeao=lambda d: d["modelo"] == campeao))
 .write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.{SCHEMA}.model_evaluation"))

print(f"Salvo: {CATALOG}.{SCHEMA}.model_evaluation | Campeão: {campeao}")
display(spark.table(f"{CATALOG}.{SCHEMA}.model_evaluation"))

## Critérios de decisão

* **Campeão:** maior AUC no teste out-of-time, com decaimento controlado vs. validação (diferença > 0,05 sugere overfitting)
* **Negócio:** se o lift do 1º–2º decisão for ≤ 1, o modelo não segmenta — rever features no CRISP-DM (voltar a 02/05)
* **Calibração:** se Brier for alto e a curva se afastar da diagonal, aplicar recalibração (Platt/isotônica) — especialmente se tiver usado RUS
* **Próximo passo:** `10_painel_report` — tabelas resumo e dashboard.